# Single Query Debug Tool

## Purpose
Interactive tool for testing retrieval on individual queries.
Useful for debugging, error analysis, and understanding method behavior.

## Inputs
- Pre-built indexes in `/work/index/`
- Query text (entered interactively or selected from dataset)

## Outputs
- Console display of top-K retrieved documents with scores
- Side-by-side comparison across methods

## What This Notebook Does
1. Lists all available retrieval methods from index registry
2. Allows selection of one or more methods to compare
3. For a given query (by item_key or free text):
   - Retrieves top-K candidates from each method
   - Displays ranked results with scores
   - Highlights correct matches
   - Shows score distributions
4. Enables quick A/B comparison between methods

In [3]:
from pathlib import Path
import json

INDEX_ROOT = Path("/work/index")

def list_methods(index_root: Path = INDEX_ROOT):
    """List available methods from /index/registry.json or folders."""
    reg_path = index_root / "registry.json"
    if reg_path.exists():
        with open(reg_path, "r", encoding="utf-8") as f:
            reg = json.load(f)
        return list(reg.keys())
    else:
        # fallback: just list subdirs
        return [p.name for p in index_root.iterdir() if p.is_dir()]

methods = list_methods()
print("Available methods:")
for i, m in enumerate(methods, 1):
    print(f"{i}. {m}")

choice = int(input("\nSelect a method by number: "))
METHOD_NAME = methods[choice-1]
print(f"\n✔ You selected: {METHOD_NAME}")


Available methods:
1. tfidf_unigram
2. tfidf_unigram_nostop
3. tfidf_unigram_phrases
4. tfidf_unigram_phrases_add
5. tfidf_unigram_phrases_replace
6. tfidf_char_3_5
7. bm25_unigram__k1-0.60__b-0.20
8. bm25_unigram__k1-0.60__b-0.35
9. bm25_unigram__k1-0.60__b-0.50
10. bm25_unigram__k1-0.60__b-0.65
11. bm25_unigram__k1-0.60__b-0.80
12. bm25_unigram__k1-0.80__b-0.20
13. bm25_unigram__k1-0.80__b-0.35
14. bm25_unigram__k1-0.80__b-0.50
15. bm25_unigram__k1-0.80__b-0.65
16. bm25_unigram__k1-0.80__b-0.80
17. bm25_unigram__k1-1.00__b-0.20
18. bm25_unigram__k1-1.00__b-0.35
19. bm25_unigram__k1-1.00__b-0.50
20. bm25_unigram__k1-1.00__b-0.65
21. bm25_unigram__k1-1.00__b-0.80
22. bm25_unigram__k1-1.20__b-0.20
23. bm25_unigram__k1-1.20__b-0.35
24. bm25_unigram__k1-1.20__b-0.50
25. bm25_unigram__k1-1.20__b-0.65
26. bm25_unigram__k1-1.20__b-0.80
27. bm25_unigram__k1-1.40__b-0.20
28. bm25_unigram__k1-1.40__b-0.35
29. bm25_unigram__k1-1.40__b-0.50
30. bm25_unigram__k1-1.40__b-0.65
31. bm25_unigram__k1-1


Select a method by number:  58



✔ You selected: bm25_unigram_params


In [4]:
# Random _resumen item: print id, parameters, and full text (no truncation)
import json, random
from pathlib import Path
import pandas as pd

COLLECTION  = "OEB"
DATA_DIR    = Path("/work/data/processed")
RESUMEN_PATH = DATA_DIR / f"{COLLECTION}_resumen.json"

def _as_record_from_obj(doc):
    """
    Robust flattening (mirrors data.ipynb): pull from doc.metadata if present,
    else fall back to top-level keys. Ensures 'parameters' is a dict.
    """
    if isinstance(doc, dict):
        text = doc.get("text") or doc.get("content")
        md   = doc.get("metadata", {}) if isinstance(doc.get("metadata"), dict) else {}
        root = doc
        doc_id = (root.get("id") or root.get("id_") or root.get("doc_id")
                  or md.get("id") or md.get("id_") or md.get("doc_id"))
    else:
        text = getattr(doc, "text", None)
        md   = getattr(doc, "metadata", None) or {}
        root = {}
        doc_id = getattr(doc, "id_", None) or getattr(doc, "doc_id", None) or md.get("id")

    def pick(key, default=None):
        return (md.get(key) if isinstance(md, dict) else None) or root.get(key, default)

    params = pick("parameters", {})
    if not isinstance(params, dict):
        params = {}

    return {
        "id": doc_id,
        "item_key": pick("item_key"),
        "parameters": params,
        "text": text if isinstance(text, str) else pick("text", ""),
        "_metadata_raw": md,  # for debugging if something is missing
    }

def load_llamaindex_docs(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, list):
        items = obj
    elif isinstance(obj, dict):
        # support {"documents":[...]}, {"data":[...]}, {"items":[...]}, {"records":[...]}
        items = None
        for key in ("documents", "data", "items", "records"):
            if key in obj and isinstance(obj[key], list):
                items = obj[key]
                break
        if items is None:
            raise TypeError(f"Unsupported JSON structure in {path}")
    else:
        raise TypeError(f"Unsupported JSON root type in {path}: {type(obj)}")
    rows = [_as_record_from_obj(d) for d in items]
    return pd.DataFrame(rows)

# Load, sample, and print
df = load_llamaindex_docs(RESUMEN_PATH)
row = df.sample(1, random_state=None).iloc[0]

print("ID:", row["id"])
print("item_key", row["item_key"])
print("Parameters:", json.dumps(row["parameters"], ensure_ascii=False, indent=2))
print("\nFull text:\n")
print(row["text"])

# Optional: if something is missing, show raw metadata to debug the shape
if (not row["parameters"]) or (row.get("item_key") in [None, ""]):
    print("\n[debug] Raw metadata for this doc:")
    print(json.dumps(row["_metadata_raw"], ensure_ascii=False, indent=2))



ID: a57ba876-2b0b-4e4a-8d80-61b5a6cc5c74
item_key OEB290aadcc
Parameters: {
  "A": {
    "label": " Nº TUBOS ",
    "values": [
      {
        "label": "a",
        "value": " 1 "
      }
    ]
  },
  "B": {
    "label": " TIPO DE TERRENO ",
    "values": [
      {
        "label": "a",
        "value": " Normal"
      }
    ]
  },
  "C": {
    "label": "TRABAJO",
    "values": [
      {
        "label": "d",
        "value": "Nocturno Excepcional"
      }
    ]
  },
  "D": {
    "label": "BANDA DE MANTENIMIENTO",
    "values": [
      {
        "label": "c",
        "value": "i < 3 horas"
      }
    ]
  },
  "F": {
    "label": "CONDICIONES DE EJECUCIÓN",
    "values": [
      {
        "label": "c",
        "value": "Cualquier condición de ejecución"
      }
    ]
  }
}

Full text:

Canalización hormigonada  1  T, polietileno libre de halógenos de 50 mm, normal. (Nocturno excepcional/i < "3" horas/Cualquier condición de ejecución)


In [7]:
# Use METHOD_NAME (selected earlier) and row["text"] (your sampled query)
import sys
from pathlib import Path
import json, importlib, numpy as np, pandas as pd

# Adjust path so Python can find /src/retrievers/*
SRC_DIR = Path("/work/src")   # change /work → your actual repo root
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

INDEX_ROOT = Path("/work/index")
index_dir = (Path(json.load(open(INDEX_ROOT/"registry.json","r",encoding="utf-8"))[METHOD_NAME]["path"])
             if (INDEX_ROOT/"registry.json").exists() else INDEX_ROOT/METHOD_NAME)

# --- load retriever for this index ---
meta = json.load(open(index_dir/"meta.json","r",encoding="utf-8"))
impl = (meta.get("impl") or meta.get("variant") or meta.get("method") or METHOD_NAME).split("__",1)[0]
retr_mod = importlib.import_module(f"retrievers.{impl}")    # exposes .load(index_dir)
searcher = retr_mod.load(str(index_dir))                    # BM25Searcher or TfidfSearcher

# --- inspect analyzer tokens and build the sparse query vector ---
text = row["text"]
analyzer = searcher.vec_q.build_analyzer()                  # frozen analyzer from the index vocab/idf
tokens   = analyzer(text)                                   # literal tokens as seen by this method  :contentReference[oaicite:0]{index=0}:contentReference[oaicite:1]{index=1}

# Sparse vector using the exact frozen vocab+idf:
if hasattr(searcher, "encode"):                             # TF–IDF retriever uses vec_q (idf frozen) :contentReference[oaicite:2]{index=2}
    Q = searcher.encode([text]).tocsr()
else:                                                       # BM25: presence -> idf, then score = Q_idf @ X_bm25.T  :contentReference[oaicite:3]{index=3}
    Q = searcher.vec_q.transform([text]).tocsr().astype(np.float32)
    Q.data = searcher.idf[Q.indices]                        # binary presence weighted by idf (query-time)

# Map non-zeros back to terms
inv_vocab = {j:t for t,j in searcher.vec_q.vocabulary_.items()}
nz_terms  = [inv_vocab[j] for j in Q.indices]
nz_vals   = Q.data

print(f"METHOD_NAME: {METHOD_NAME}  (impl={impl})")
print("\n=== QUERY TEXT (full) ===\n", text)

print("\n=== ANALYZER TOKENS ===")
print(tokens[:200] if len(tokens) > 200 else tokens)
if len(tokens) > 200:
    print(f"... ({len(tokens)-200} more tokens)")

print("\n=== SPARSE VECTOR NON-ZEROS (term -> value) ===")
pairs = sorted(zip(nz_terms, nz_vals), key=lambda x: x[0])
for t, v in pairs[:300]:
    print(f"{t}\t{float(v):.6f}")
if len(pairs) > 300:
    print(f"... ({len(pairs)-300} more)")

# --- quick retrieval to prove it runs with the same encoding ---
top_idx, top_s = searcher.search(text, k=5)
ext_ids = np.asarray(searcher.external_ids, dtype=object)
top_keys = [str(ext_ids[i]) for i in top_idx]
print("\n=== TOP-5 ===")
for r, (k, s) in enumerate(zip(top_keys, top_s), start=1):
    print(f"{r}. {k}\t{s:.6f}")


METHOD_NAME: bm25_unigram_params  (impl=bm25_unigram_params)

=== QUERY TEXT (full) ===
 Canalización hormigonada  1  T, polietileno libre de halógenos de 50 mm, normal. (Nocturno excepcional/i < "3" horas/Cualquier condición de ejecución)

=== ANALYZER TOKENS ===
['canalizacion', 'hormigonada', '1', 't', 'polietileno', 'libre', 'de', 'halogenos', 'de', '50', 'mm', 'normal', 'nocturno', 'excepcional', 'i', '3', 'horas', 'cualquier', 'condicion', 'de', 'ejecucion']

=== SPARSE VECTOR NON-ZEROS (term -> value) ===
1	2.151470
3	0.602024
50	2.014698
canalizacion	0.029832
condicion	1.099212
cualquier	0.497996
de	0.000011
ejecucion	0.000621
excepcional	0.693000
halogenos	0.223004
horas	0.290070
hormigonada	0.106907
i	0.289901
libre	0.223004
mm	0.094409
nocturno	1.099844
polietileno	0.218281

=== TOP-5 ===
1. OEB290aadcc	12.727932
2. OEB290aadbc	12.709252
3. OEB290acdcc	12.652603
4. OEB290acdbc	12.633885
5. OEB290aedcc	12.596618


In [8]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/work/data/processed")
COLLECTION = "OEB"

# Load long-format features
long_path = DATA_DIR / f"{COLLECTION}_long_feats.parquet"
df_long = pd.read_parquet(long_path)

# First candidate external_id (item_key)
cand_key = top_keys[0]
print("First candidate item_key:", cand_key)

# Lookup in long-format df
row_long = df_long[df_long["item_key"] == cand_key].iloc[0]

print("\n=== Full long-format text ===\n")
print(row_long["text"])

print("\n=== text_word_params ===\n")
print(row_long["text_word_params"])


First candidate item_key: OEB290aadcc

=== Full long-format text ===

Canalización hormigonada de  1  tubos de polietileno libre de halógenos de 50 mm de diámetro en cualquier clase de terreno, excepto roca, incluso  el relleno y compactado de la zanja,  el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo  Trabajo: Nocturno Excepcional Banda de mantenimiento: i < 3 horas Condiciones de ejecución: Cualquier condición de ejecución

=== text_word_params ===

canalización hormigonada de 1 tubos de polietileno libre de halógenos de 50 mm de diámetro en cualquier clase de terreno excepto roca incluso el relleno y compactado de la zanja el suministro y montaje de los tubos y hormigón tipo he 20 sin vibrar la prueba de los conductos el transporte y la retirada de los productos al lugar de empleo trabajo nocturno excepcional banda de mantenimiento i 3 horas condiciones de ejecución cua

In [10]:
# Side-by-side experiment: compare top-5 results for the SAME query text
# between bm25_unigram and bm25_unigram_params indices.
#
# Assumptions:
# - You already sampled a query: `row["text"]` (plain/original text)
# - Built indexes exist under /work/index for both variants.
# - Your project code is under /work/src (so we add it to sys.path).
#
# What this cell shows:
# 1) Resolves index dirs for bm25_unigram (plain) and bm25_unigram_params (with param_* tokens).
# 2) Loads both retrievers and runs search on the SAME query_text.
# 3) Displays a side-by-side top-5 table (rank and score from each method).
# 4) Prints quick diagnostics: vocab sizes and which text_field each index uses.
#
from pathlib import Path
import sys, json, importlib
import numpy as np
import pandas as pd

# --- Config/paths ---
WORK = Path("/work")
SRC  = WORK / "src"
INDEX_ROOT = WORK / "index"
DATA_DIR = WORK / "data" / "processed"
COLLECTION = "OEB"

# Ensure /work/src is importable (retrievers.*)
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

def read_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def discover_index_by_params_flag(want_params: bool):
    """
    Scan /work/index and return the FIRST index dir whose impl family is bm25_unigram
    and whose fields.json 'text_field' ends with '_params' iff want_params=True.
    Fallbacks: match on folder name containing 'bm25_unigram' and 'params'.
    """
    if not INDEX_ROOT.exists():
        raise FileNotFoundError(f"{INDEX_ROOT} not found")

    candidates = []
    for d in sorted(INDEX_ROOT.iterdir()):
        if not d.is_dir():
            continue
        meta = d / "meta.json"
        fields = d / "fields.json"
        if not meta.exists() or not fields.exists():
            continue
        m = read_json(meta)
        f = read_json(fields)
        impl = (m.get("impl") or m.get("method") or "").lower()
        tfield = f.get("text_field", "")
        is_bm25_uni = "bm25_unigram" in impl or "bm25" in m.get("method","").lower()
        has_params = tfield.endswith("_params")
        if is_bm25_uni:
            candidates.append((d, impl, tfield, has_params))

    # Prefer explicit text_field match
    for d, impl, tfield, has_params in candidates:
        if has_params == want_params:
            return d, impl, tfield

    # Fallback: name contains marker
    for d, impl, tfield, has_params in candidates:
        name = d.name.lower()
        if want_params and ("params" in name):
            return d, impl, tfield
        if (not want_params) and ("params" not in name):
            return d, impl, tfield

    raise RuntimeError("Could not find a matching index for want_params=" + str(want_params))

def load_searcher(index_dir: Path):
    meta = read_json(index_dir / "meta.json")
    impl = (meta.get("impl") or meta.get("method") or index_dir.name).split("__",1)[0]
    try:
        retr_mod = importlib.import_module(f"retrievers.{impl}")
    except ModuleNotFoundError:
        # Some builds may record impl as 'bm25_unigram_params' but share the same class
        if impl.startswith("bm25_unigram"):
            retr_mod = importlib.import_module("retrievers.bm25_unigram")
        else:
            raise
    # module must expose load(index_dir) -> searcher
    searcher = retr_mod.load(str(index_dir))
    return impl, searcher

# --- Resolve both indices ---
idx_plain_dir, impl_plain, tfield_plain = discover_index_by_params_flag(False)
idx_params_dir, impl_params, tfield_params = discover_index_by_params_flag(True)

# --- Load both searchers ---
implA, searcherA = load_searcher(idx_plain_dir)
implB, searcherB = load_searcher(idx_params_dir)

# --- Use the SAME query text the user sampled earlier ---
try:
    query_text = row["text"]
except Exception as e:
    raise RuntimeError("Expected a previously-sampled query in row['text']. Please define it first.") from e

# --- Run both searches ---
k = 5
top_idx_A, top_s_A = searcherA.search(query_text, k=k)
top_idx_B, top_s_B = searcherB.search(query_text, k=k)

extA = np.asarray(searcherA.external_ids, dtype=object)
extB = np.asarray(searcherB.external_ids, dtype=object)

top_keys_A = [str(extA[i]) for i in top_idx_A]
top_keys_B = [str(extB[i]) for i in top_idx_B]

dfA = pd.DataFrame({"rank_A": np.arange(1, k+1), "item_key": top_keys_A, "score_A": top_s_A})
dfB = pd.DataFrame({"rank_B": np.arange(1, k+1), "item_key": top_keys_B, "score_B": top_s_B})

# side-by-side merge by item_key to see overlaps and differences
merged = dfA.merge(dfB, on="item_key", how="outer").sort_values(["rank_A","rank_B"], na_position="last")
merged.reset_index(drop=True, inplace=True)

# --- Diagnostics ---
meta_plain = read_json(idx_plain_dir / "meta.json")
meta_params = read_json(idx_params_dir / "meta.json")
vocab_plain = read_json(idx_plain_dir / "data" / "vocab.json")
vocab_params = read_json(idx_params_dir / "data" / "vocab.json")

print("Query text:\n", query_text)
print("\nIndex A (plain):", idx_plain_dir)
print(" - impl:", implA, "| text_field:", tfield_plain, "| vocab_size:", len(vocab_plain))
print("Index B (params):", idx_params_dir)
print(" - impl:", implB, "| text_field:", tfield_params, "| vocab_size:", len(vocab_params))

# --- Show the table to the user ---
display("Top-5 side-by-side: bm25_unigram vs bm25_unigram_params", merged)

# Also return merged so it prints in the output cell
merged


Query text:
 Canalización hormigonada  1  T, polietileno libre de halógenos de 50 mm, normal. (Nocturno excepcional/i < "3" horas/Cualquier condición de ejecución)

Index A (plain): /work/index/bm25_unibigram
 - impl: bm25_unibigram | text_field: text_word | vocab_size: 1142
Index B (params): /work/index/bm25_unigram_params
 - impl: bm25_unigram_params | text_field: text_word_params | vocab_size: 398


'Top-5 side-by-side: bm25_unigram vs bm25_unigram_params'

,rank_A,item_key,score_A,rank_B,score_B
0,1.0,OEB290aadcc,30.826994,1.0,12.727932
1,2.0,OEB290acdcc,30.699100,3.0,12.652603
2,3.0,OEB290aedcc,30.527386,5.0,12.596618
3,4.0,OEB290addcc,30.413988,NaN,NaN
4,5.0,OEB290agdcc,30.078770,NaN,NaN
5,NaN,OEB290aadbc,NaN,2.0,12.709252
6,NaN,OEB290acdbc,NaN,4.0,12.633885


,rank_A,item_key,score_A,rank_B,score_B
0,1.0,OEB290aadcc,30.826994,1.0,12.727932
1,2.0,OEB290acdcc,30.699100,3.0,12.652603
2,3.0,OEB290aedcc,30.527386,5.0,12.596618
3,4.0,OEB290addcc,30.413988,NaN,NaN
4,5.0,OEB290agdcc,30.078770,NaN,NaN
5,NaN,OEB290aadbc,NaN,2.0,12.709252
6,NaN,OEB290acdbc,NaN,4.0,12.633885


In [11]:
# Find a fair pair: bm25 UNIGRAM vs bm25 UNIGRAM_PARAMS with identical k1/b/analyzer settings
from pathlib import Path
import json

INDEX_ROOT = Path("/work/index")

def j(p): 
    with open(p, "r", encoding="utf-8") as f: 
        return json.load(f)

def fm(index_dir: Path):
    meta = j(index_dir / "meta.json")
    fields = j(index_dir / "fields.json")
    impl = (meta.get("impl") or meta.get("method") or index_dir.name).split("__",1)[0]
    p = meta.get("params", {})
    sig = {
        "impl_base": "bm25_unigram",             # we want unigram only
        "analyzer": p.get("analyzer","word"),
        "ngram_range": tuple(p.get("ngram_range",[1,1])),
        "lowercase": p.get("lowercase", True),
        "token_pattern": p.get("token_pattern", r"(?u)\b\w+\b"),
        "strip_accents": p.get("strip_accents", "unicode"),
        "k1": float(p.get("k1", 1.2)),
        "b":  float(p.get("b", 0.75)),
    }
    return {
        "dir": index_dir, 
        "impl": impl, 
        "text_field": fields.get("text_field",""),
        "sig": sig
    }

# collect bm25 indices
cands = []
for d in sorted(INDEX_ROOT.iterdir()):
    if not d.is_dir(): 
        continue
    if not (d/"meta.json").exists() or not (d/"fields.json").exists():
        continue
    try:
        rec = fm(d)
    except Exception:
        continue
    if "bm25" in rec["impl"]:
        cands.append(rec)

# split plain vs params (unigram only)
plain = [c for c in cands if c["sig"]["impl_base"]=="bm25_unigram" 
         and c["sig"]["ngram_range"]==(1,1) and not c["text_field"].endswith("_params")]
params = [c for c in cands if c["sig"]["impl_base"]=="bm25_unigram" 
          and c["sig"]["ngram_range"]==(1,1) and c["text_field"].endswith("_params")]

# try to match by exact BM25 signature (except text_field)
def key_without_field(c): 
    k=c["sig"].copy() ; return tuple(sorted(k.items()))

pairs = []
for p in plain:
    for q in params:
        if key_without_field(p)==key_without_field(q):
            pairs.append((p,q))

if pairs:
    A,B = pairs[0]
    print("✅ Found a fair pair (only text_field differs):")
    print("A:", A["dir"], "| text_field:", A["text_field"], "| sig:", A["sig"])
    print("B:", B["dir"], "| text_field:", B["text_field"], "| sig:", B["sig"])
else:
    print("❌ No fair unigram pair found. Here’s what exists (showing first few):\n")
    for z in (plain[:3]+params[:3]):
        print(z["dir"], "| text_field:", z["text_field"], "| sig:", z["sig"])


✅ Found a fair pair (only text_field differs):
A: /work/index/bm25_unigram | text_field: text_word | sig: {'impl_base': 'bm25_unigram', 'analyzer': 'word', 'ngram_range': (1, 1), 'lowercase': True, 'token_pattern': '(?u)\\b\\w+\\b', 'strip_accents': 'unicode', 'k1': 0.8, 'b': 0.35}
B: /work/index/bm25_unigram_params__k1-0.80__b-0.35 | text_field: text_word_params | sig: {'impl_base': 'bm25_unigram', 'analyzer': 'word', 'ngram_range': (1, 1), 'lowercase': True, 'token_pattern': '(?u)\\b\\w+\\b', 'strip_accents': 'unicode', 'k1': 0.8, 'b': 0.35}


In [12]:
# Fair A/B: same BM25 (k1=0.8, b=0.35, word unigrams), only text_field differs
from pathlib import Path
import sys, json, importlib
import numpy as np
import pandas as pd


# --- plug your pair here (from your printout) ---
A_DIR = Path("/work/index/bm25_unigram")                       # text_word
B_DIR = Path("/work/index/bm25_unigram_params__k1-0.80__b-0.35") # text_word_params

# ensure retrievers are importable
SRC = Path("/work/src")
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

def j(p): 
    with open(p, "r", encoding="utf-8") as f: 
        return json.load(f)

def load_searcher(index_dir: Path):
    meta = j(index_dir / "meta.json")
    impl = (meta.get("impl") or meta.get("method") or index_dir.name).split("__",1)[0]
    try:
        mod = importlib.import_module(f"retrievers.{impl}")
    except ModuleNotFoundError:
        # bm25_unigram_params may just reuse bm25_unigram class
        mod = importlib.import_module("retrievers.bm25_unigram")
    return mod.load(str(index_dir)), meta

# load both
searcherA, metaA = load_searcher(A_DIR)
searcherB, metaB = load_searcher(B_DIR)
vocabA = j(A_DIR/"data"/"vocab.json")
vocabB = j(B_DIR/"data"/"vocab.json")
fieldA = j(A_DIR/"fields.json").get("text_field","")
fieldB = j(B_DIR/"fields.json").get("text_field","")

# query text already sampled
qtext = row["text"]

# run
K = 10
idxA, scoreA = searcherA.search(qtext, k=K)
idxB, scoreB = searcherB.search(qtext, k=K)
extA = np.asarray(searcherA.external_ids, dtype=object)
extB = np.asarray(searcherB.external_ids, dtype=object)
keysA = [str(extA[i]) for i in idxA]
keysB = [str(extB[i]) for i in idxB]

# side-by-side
dfA = pd.DataFrame({"rank_A": np.arange(1, len(keysA)+1), "item_key": keysA, "score_A": scoreA})
dfB = pd.DataFrame({"rank_B": np.arange(1, len(keysB)+1), "item_key": keysB, "score_B": scoreB})
merged = dfA.merge(dfB, on="item_key", how="outer").sort_values(["rank_A","rank_B"], na_position="last").reset_index(drop=True)

print("Query text:\n", qtext, "\n")
print("A:", A_DIR, "| text_field:", fieldA, "| vocab_size:", len(vocabA))
print("B:", B_DIR, "| text_field:", fieldB, "| vocab_size:", len(vocabB))

# simple overlap stats
setA, setB = set(keysA), set(keysB)
overlap = sorted(setA & setB, key=lambda k: (keysA.index(k), keysB.index(k)))
jacc = len(setA & setB) / len(setA | setB) if (setA | setB) else 0.0
print(f"\nTop-{K} overlap: {len(overlap)} items | Jaccard={jacc:.3f}")
if overlap:
    print("Overlap (in A-rank,B-rank):")
    for k in overlap[:10]:
        print(f"  {k}: {keysA.index(k)+1}, {keysB.index(k)+1}")

# show table
display("A vs B (Top-10): bm25_unigram vs bm25_unigram_params", merged)
merged.head(20)


Query text:
 Canalización hormigonada  1  T, polietileno libre de halógenos de 50 mm, normal. (Nocturno excepcional/i < "3" horas/Cualquier condición de ejecución) 

A: /work/index/bm25_unigram | text_field: text_word | vocab_size: 337
B: /work/index/bm25_unigram_params__k1-0.80__b-0.35 | text_field: text_word_params | vocab_size: 398

Top-10 overlap: 10 items | Jaccard=1.000
Overlap (in A-rank,B-rank):
  OEB290aadcc: 1, 1
  OEB290aadbc: 2, 2
  OEB290acdcc: 3, 3
  OEB290acdbc: 4, 4
  OEB290aedcc: 5, 5
  OEB290aedbc: 6, 6
  OEB290addcc: 7, 7
  OEB290addbc: 8, 8
  OEB290agdcc: 9, 10
  OEB290abdcc: 10, 9


'A vs B (Top-10): bm25_unigram vs bm25_unigram_params'

,rank_A,item_key,score_A,rank_B,score_B
0,1,OEB290aadcc,12.778169,1,12.767624
1,2,OEB290aadbc,12.754571,2,12.745371
2,3,OEB290acdcc,12.684243,3,12.675011
3,4,OEB290acdbc,12.660648,4,12.652754
4,5,OEB290aedcc,12.613721,5,12.608481
5,6,OEB290aedbc,12.590385,6,12.586459
6,7,OEB290addcc,12.567142,7,12.564515
7,8,OEB290addbc,12.543978,8,12.542645
8,9,OEB290agdcc,12.429439,10,12.434435
9,10,OEB290abdcc,12.429439,9,12.434435


,rank_A,item_key,score_A,rank_B,score_B
0,1,OEB290aadcc,12.778169,1,12.767624
1,2,OEB290aadbc,12.754571,2,12.745371
2,3,OEB290acdcc,12.684243,3,12.675011
3,4,OEB290acdbc,12.660648,4,12.652754
4,5,OEB290aedcc,12.613721,5,12.608481
5,6,OEB290aedbc,12.590385,6,12.586459
6,7,OEB290addcc,12.567142,7,12.564515
7,8,OEB290addbc,12.543978,8,12.542645
8,9,OEB290agdcc,12.429439,10,12.434435
9,10,OEB290abdcc,12.429439,9,12.434435
